# FASHN VTON v1.5 Colab Eval

Notebook này chạy trực tiếp trên Google Colab để test `avatar image + garment image + category` trước khi quyết định build Replicate Deployment.

Runtime nên chọn: `Runtime > Change runtime type > T4 / A100 / L4 GPU`.

In [ ]:
!nvidia-smi
import torch
print('CUDA available:', torch.cuda.is_available())
print('CUDA device:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'none')

## Install FASHN VTON

Official flow: clone repo, `pip install -e .`, then `python scripts/download_weights.py --weights-dir ./weights`.

In [ ]:
%cd /content
!test -d fashn-vton-1.5 || git clone https://github.com/fashn-AI/fashn-vton-1.5.git
%cd /content/fashn-vton-1.5
!pip install -q -e .

In [ ]:
%cd /content/fashn-vton-1.5
!python scripts/download_weights.py --weights-dir ./weights

## Upload Inputs

Upload 2 file theo thứ tự:

- avatar image: synthetic full/upper body avatar
- garment image: product garment

Nếu đã mount Google Drive, có thể bỏ qua upload và set path thủ công ở cell kế tiếp.

In [ ]:
from google.colab import files
uploaded = files.upload()
print('Uploaded:', list(uploaded.keys()))

## Configure Case

`CATEGORY` phải là một trong: `tops`, `bottoms`, `one-pieces`.

`RUN_VARIANTS` quyết định cách FASHN chạy. Với garment ảnh sản phẩm đặt phẳng, giữ `garment_photo_type='flat-lay'`.

In [ ]:
from pathlib import Path

AVATAR_STRATEGY = 'curated_template'
ENABLE_GARMENT_CENTER_CROP = False
GARMENT_CROP_BOX = None  # example: (120, 80, 900, 930) as left, top, right, bottom
uploaded_names = list(globals().get('uploaded', {}).keys())

if len(uploaded_names) >= 2:
    # Assumption: upload avatar first, garment second.
    AVATAR_IMAGE = Path(uploaded_names[0]).resolve()
    GARMENT_IMAGE = Path(uploaded_names[1]).resolve()
else:
    # If using Drive or manually copied files, edit these two paths.
    AVATAR_IMAGE = Path('/content/fashn-vton-1.5/avatar.png')
    GARMENT_IMAGE = Path('/content/fashn-vton-1.5/garment.png')

CASE_ID = 'fashn-avatar-case-001'
CATEGORY = 'tops'  # tops | bottoms | one-pieces
GARMENT_TYPE = 'jersey'
GARMENT_SLEEVE_LENGTH = 'short_sleeve'
GARMENT_REGION = 'upper_body'
AVATAR_FRAMING = 'full_body'
RUN_VARIANTS = [
    {
        'run_id': 'fashn_flatlay_segfree_s30_g15',
        'garment_photo_type': 'flat-lay',
        'num_timesteps': 30,
        'guidance_scale': 1.5,
        'seed': 42,
        'segmentation_free': True,
    },
    # Uncomment one extra run only after the clean-garment baseline is inspected.
    # {
    #     'run_id': 'fashn_flatlay_masked_s30_g15',
    #     'garment_photo_type': 'flat-lay',
    #     'num_timesteps': 30,
    #     'guidance_scale': 1.5,
    #     'seed': 42,
    #     'segmentation_free': False,
    # },
]

assert CATEGORY in {'tops', 'bottoms', 'one-pieces'}
for variant in RUN_VARIANTS:
    assert variant['garment_photo_type'] in {'flat-lay', 'model'}
print('Avatar:', AVATAR_IMAGE, AVATAR_IMAGE.exists())
print('Garment:', GARMENT_IMAGE, GARMENT_IMAGE.exists())
print('Planned variants:', [variant['run_id'] for variant in RUN_VARIANTS])

Nếu file upload chưa nằm đúng tên `avatar.png` / `garment.png`, chạy cell này để copy từ uploaded file sang tên chuẩn.

In [ ]:
import shutil

# Edit only if needed.
UPLOADED_AVATAR_FILENAME = ''
UPLOADED_GARMENT_FILENAME = ''

if UPLOADED_AVATAR_FILENAME:
    shutil.copy(Path('/content/fashn-vton-1.5') / UPLOADED_AVATAR_FILENAME, AVATAR_IMAGE)
if UPLOADED_GARMENT_FILENAME:
    shutil.copy(Path('/content/fashn-vton-1.5') / UPLOADED_GARMENT_FILENAME, GARMENT_IMAGE)

print('Avatar:', AVATAR_IMAGE, AVATAR_IMAGE.exists())
print('Garment:', GARMENT_IMAGE, GARMENT_IMAGE.exists())

## Optional: Clean Garment Crop

Nếu garment image có nhiều background/clutter, bật `ENABLE_GARMENT_CENTER_CROP` hoặc set `GARMENT_CROP_BOX`. Avatar nên là curated template đã QC thủ công.

In [ ]:
from PIL import Image

output_dir = Path('/content/fashn-vton-eval-outputs')
output_dir.mkdir(parents=True, exist_ok=True)

if GARMENT_CROP_BOX is not None or ENABLE_GARMENT_CENTER_CROP:
    garment_for_crop = Image.open(GARMENT_IMAGE).convert('RGB')
    if GARMENT_CROP_BOX is None:
        width, height = garment_for_crop.size
        margin_x = int(width * 0.08)
        margin_y = int(height * 0.08)
        GARMENT_CROP_BOX = (margin_x, margin_y, width - margin_x, height - margin_y)
    cropped_garment = garment_for_crop.crop(GARMENT_CROP_BOX)
    cropped_path = output_dir / f'{CASE_ID}-garment-crop.png'
    cropped_garment.save(cropped_path)
    GARMENT_IMAGE = cropped_path
    print('Using cropped garment:', GARMENT_IMAGE, GARMENT_CROP_BOX)
else:
    print('Using garment without crop:', GARMENT_IMAGE)

print('Using curated avatar template:', AVATAR_IMAGE)

In [ ]:
from PIL import Image
import matplotlib.pyplot as plt

avatar = Image.open(AVATAR_IMAGE).convert('RGB')
garment = Image.open(GARMENT_IMAGE).convert('RGB')

fig, axes = plt.subplots(1, 2, figsize=(12, 6))
axes[0].imshow(avatar)
axes[0].set_title('Avatar')
axes[0].axis('off')
axes[1].imshow(garment)
axes[1].set_title('Garment')
axes[1].axis('off')
plt.show()

## Run Try-On

In [ ]:
import time
from fashn_vton import TryOnPipeline

pipeline = TryOnPipeline(weights_dir='./weights')
output_dir = Path('/content/fashn-vton-eval-outputs')
output_dir.mkdir(parents=True, exist_ok=True)
generated_records = []

for variant in RUN_VARIANTS:
    print('Running variant:', variant['run_id'])
    started = time.perf_counter()
    result = pipeline(
        person_image=avatar,
        garment_image=garment,
        category=CATEGORY,
        garment_photo_type=variant['garment_photo_type'],
        num_samples=1,
        num_timesteps=variant['num_timesteps'],
        guidance_scale=variant['guidance_scale'],
        seed=variant['seed'],
        segmentation_free=variant['segmentation_free'],
    )
    latency_seconds = time.perf_counter() - started
    generated_path = output_dir / f"{CASE_ID}-{variant['run_id']}.png"
    result.images[0].save(generated_path)
    generated_records.append({
        'variant': variant,
        'path': generated_path,
        'latency_seconds': latency_seconds,
    })
    print('Saved:', generated_path)
    print('Latency seconds:', latency_seconds)

In [ ]:
num_results = len(generated_records)
fig, axes = plt.subplots(1, 2 + num_results, figsize=(6 * (2 + num_results), 7))
axes[0].imshow(avatar)
axes[0].set_title('Avatar')
axes[0].axis('off')
axes[1].imshow(garment)
axes[1].set_title('Garment')
axes[1].axis('off')
for index, record in enumerate(generated_records, start=2):
    generated = Image.open(record['path']).convert('RGB')
    axes[index].imshow(generated)
    axes[index].set_title(record['variant']['run_id'])
    axes[index].axis('off')
plt.show()

## Write Eval Report

Report này bám theo avatar preview eval format trong app để mình có thể so sánh với các report trước.

In [ ]:
import hashlib
import json
from datetime import datetime, timezone

def file_sha256(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open('rb') as file:
        for chunk in iter(lambda: file.read(1024 * 1024), b''):
            digest.update(chunk)
    return digest.hexdigest()

started_at = datetime.now(timezone.utc).isoformat()
avatar_hash = file_sha256(AVATAR_IMAGE)
garment_hash = file_sha256(GARMENT_IMAGE)
preview_results = []
for record in generated_records:
    generated_path = record['path']
    variant = record['variant']
    preview_results.append(
        {
            'run_id': variant['run_id'],
            'status': 'succeeded',
            'error': None,
            'model': 'fashn-ai/fashn-vton-1.5-local',
            'input_mapping': 'local_fashn_vton',
            'prompt_version': 'fashn-vton-local-v1',
            'fashn_category': CATEGORY,
            'preview_context_prompt_sha256': None,
            'latency_seconds': record['latency_seconds'],
            'generated_image_bytes': generated_path.stat().st_size,
            'generated_image_path': str(generated_path),
            'generated_image_sha256': file_sha256(generated_path),
            'preview_cache_key': None,
            'parameters': {
                'garment_photo_type': variant['garment_photo_type'],
                'num_timesteps': variant['num_timesteps'],
                'guidance_scale': variant['guidance_scale'],
                'seed': variant['seed'],
                'segmentation_free': variant['segmentation_free'],
            },
        }
    )

report = {
    'started_at': started_at,
    'finished_at': datetime.now(timezone.utc).isoformat(),
    'summary': {
        'total_cases': 1,
        'preview_runs_per_case': len(preview_results),
        'total_preview_runs': len(preview_results),
        'succeeded': len(preview_results),
        'failed': 0,
    },
    'cases': [
        {
            'case_id': CASE_ID,
            'status': 'succeeded',
            'error': None,
            'input_hashes': {
                'avatar_image_sha256': avatar_hash,
                'product_image_sha256': garment_hash,
            },
            'avatar_profile': None,
            'avatar_prompt_version': None,
            'garment_type': GARMENT_TYPE,
            'garment_sleeve_length': GARMENT_SLEEVE_LENGTH,
            'garment_region': GARMENT_REGION,
            'fashn_category': CATEGORY,
            'avatar_framing': AVATAR_FRAMING,
            'avatar_image': {
                'source': AVATAR_STRATEGY,
                'path': str(AVATAR_IMAGE),
                'cache_hit': None,
                'cache_key': None,
                'prompt_version': None,
                'prompt_sha256': None,
                'model': None,
                'avatar_catalog_version': None,
                'framing': AVATAR_FRAMING,
                'strategy': AVATAR_STRATEGY,
            },
            'garment_preprocessing': {
                'crop_enabled': bool(GARMENT_CROP_BOX is not None),
                'crop_box': GARMENT_CROP_BOX,
                'final_garment_path': str(GARMENT_IMAGE),
            },
            'preview_results': preview_results,
            'total_latency_seconds': sum(record['latency_seconds'] for record in generated_records),
            'manual_quality_notes': {
                'garment_preservation': '',
                'body_similarity': '',
                'identity_privacy': '',
                'overall': '',
            },
        }
    ],
}

report_path = output_dir / f'{CASE_ID}-report.json'
report_path.write_text(json.dumps(report, indent=2, sort_keys=True), encoding='utf-8')
print('Report written to', report_path)
print(json.dumps(report['summary'], indent=2, sort_keys=True))

In [ ]:
from google.colab import files
for record in generated_records:
    files.download(str(record['path']))
files.download(str(report_path))